# Day 3 — RAGAS Evaluation & LangSmith

Evaluation-driven RAG development: define a golden dataset \u2192 run pipeline \u2192 score on faithfulness/relevancy/precision/recall \u2192 establish baseline \u2192 change one thing \u2192 compare.
This is the engineering discipline that separates production RAG from demos.
Used by teams at Tredence, Synechron, and Hexaware to gate production deployments.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
from day3.evaluation import (
    QAPair, EvalResult, GOLDEN_DATASET,
    compute_ragas_proxy, RAGEvaluator, ab_test, chunking_ablation
)
from day3.rag_pipeline import build_product_rag

def mock_embed(texts):
    vecs = []
    for t in texts:
        rng = np.random.default_rng(abs(hash(t)) % (2**31))
        v = rng.standard_normal(384).astype(np.float32)
        v = v / np.linalg.norm(v)
        vecs.append(v)
    return np.array(vecs)

print(f"Golden dataset: {len(GOLDEN_DATASET)} Q&A pairs")
for qa in GOLDEN_DATASET[:3]:
    print(f"  [{qa.category}] {qa.question}")

## 1. RAGAS Metrics

Four metrics measure different aspects of RAG quality.
Faithfulness (is the answer grounded in context?), Answer Relevancy (does the answer address the question?),
Context Precision (signal-to-noise ratio of retrieved chunks), Context Recall (did retrieval find all needed info?).

In [ ]:
# Run a single evaluation
result = compute_ragas_proxy(
    question     = "What is the best laptop for machine learning?",
    answer       = "MacBook Pro 14 with M3 Pro chip and 18GB memory is ideal for ML workloads.",
    context      = "MacBook Pro 14 M3 Pro chip with 18GB unified memory, 18-hour battery life, ideal for machine learning and data science.",
    ground_truth = "MacBook Pro 14 M3 Pro (\u20b91,99,900) with M3 Pro chip, 18GB unified memory is best for ML.",
)

print(f"Question:    {result.question[:60]}")
print(f"Faithfulness:      {result.faithfulness:.4f}  (\u22650.85 = pass)")
print(f"Answer Relevancy:  {result.answer_relevancy:.4f}  (\u22650.70 = pass)")
print(f"Context Precision: {result.context_precision:.4f}")
print(f"Context Recall:    {result.context_recall:.4f}")
print(f"Overall average:   {result.average:.4f}")
print(f"PASSED: {result.passed}")

## 2. Full Evaluation on Golden Dataset

Build the pipeline and evaluate it on all 10 golden Q&A pairs.
The RAGEvaluator.run() method processes each question, scores it, and returns structured EvalResult objects.

In [ ]:
pipeline = build_product_rag(mock_embed_fn=mock_embed)
evaluator = RAGEvaluator(pipeline)

print("Running evaluation on 10 golden questions...")
results = evaluator.run(GOLDEN_DATASET)
print(f"Evaluated {len(results)} questions")

## 3. Evaluation Summary

Aggregate statistics across all questions.
Pass rate tells you the fraction of questions where both faithfulness \u2265 0.85 and answer_relevancy \u2265 0.70.
Aim for >80% pass rate in production.

In [ ]:
summary = evaluator.summary(results)

print("Evaluation Summary:")
print("-" * 60)
for metric in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    m = summary[metric]
    print(f"  {metric:<22}: mean={m['mean']:.4f}  std={m['std']:.4f}  "
          f"min={m['min']:.4f}  max={m['max']:.4f}")

print(f"\nPass rate: {summary['pass_rate']:.1%} "
      f"({summary['num_passed']}/{summary['num_evaluated']} questions passed)")

# Per-question results
print("\nPer-question results:")
for r in results:
    status = "\u2713" if r.passed else "\u2717"
    print(f"  [{status}] faith={r.faithfulness:.3f} rel={r.answer_relevancy:.3f} | {r.question[:55]}")

## 4. Failing Questions

Identify which questions fail quality thresholds. These are your debugging targets.
Low faithfulness means the answer has hallucinated content; low relevancy means the answer doesn't address the question.

In [ ]:
failing = evaluator.failing_questions(results)
print(f"Failing questions: {len(failing)}/{len(results)}")
for r in failing:
    print(f"\n  Question: {r.question}")
    print(f"  Answer: {r.answer[:100]}")
    print(f"  faith={r.faithfulness:.3f} rel={r.answer_relevancy:.3f}")
    
if not failing:
    print("All questions passed! (mock embeddings produce consistent results)")

## 5. Save Baseline

Save the evaluation results as a JSON baseline. Commit this to Git.
Future pipeline changes must beat this baseline \u2014 if scores drop, you've found a regression.

In [ ]:
baseline = evaluator.save_baseline(results, "day3_baseline.json")
print("Baseline saved to day3_baseline.json")
print(f"Key metrics: faithfulness={baseline['summary']['faithfulness']['mean']:.4f}, "
      f"pass_rate={baseline['summary']['pass_rate']:.1%}")

## 6. A/B Test: Hybrid vs Dense-Only

Compare two pipeline configurations on the same golden questions.
The A/B test reveals whether hybrid retrieval actually improves scores over dense-only.
One change, one evaluation, measurable result.

In [ ]:
# Pipeline A: full hybrid + reranker
pipeline_a = build_product_rag(mock_embed_fn=mock_embed)

# Pipeline B: dense-only (no BM25, no reranker)
from day3.rag_pipeline import AdvancedRAGPipeline
from day3.hybrid_search import generate_product_corpus
pipeline_b = AdvancedRAGPipeline(
    use_hybrid=False, use_reranker=False, use_cache=False,
    top_k=3, rerank_top_k=3, mock_embed_fn=mock_embed
)
corpus = generate_product_corpus()
pipeline_b.index([{"text": t, "metadata": {}} for t in corpus])

print("Running A/B test (5 questions)...")
ab_results = ab_test(
    pipeline_a, pipeline_b,
    GOLDEN_DATASET[:5],
    name_a="hybrid_rrf", name_b="dense_only"
)

print(f"\nOverall winner: {ab_results['overall_winner']}")
print("\nPer-metric winners:")
for metric, winner in ab_results["metric_winners"].items():
    a_mean = ab_results["hybrid_rrf"]["summary"][metric]["mean"]
    b_mean = ab_results["dense_only"]["summary"][metric]["mean"]
    print(f"  {metric:<22}: {winner:<15} (hybrid={a_mean:.4f} vs dense={b_mean:.4f})")

## 7. Real RAGAS (Production)

Production RAGAS uses an LLM judge (GPT-4) instead of proxies.
When you have an OPENAI_API_KEY, this gives much more accurate faithfulness and relevancy scores.

In [ ]:
print("""
PRODUCTION RAGAS (requires OPENAI_API_KEY and: pip install ragas):

  from ragas import evaluate
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
  from datasets import Dataset

  dataset = Dataset.from_dict({
      "question":   [qa.question for qa in GOLDEN_DATASET],
      "answer":     [result.answer for result in rag_results],
      "contexts":   [[result.reranked_chunks] for result in rag_results],
      "ground_truth": [qa.ground_truth for qa in GOLDEN_DATASET],
  })
  
  scores = evaluate(
      dataset,
      metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
  )
  print(scores)

LANGSMITH INTEGRATION (requires LANGCHAIN_API_KEY):
  
  export LANGCHAIN_API_KEY=ls-xxxx
  export LANGCHAIN_TRACING_V2=true
  export LANGCHAIN_PROJECT=day3-advanced-rag
  
  # All LangChain calls are now automatically traced at smith.langchain.com
""")